# p4-onset.ipynb — 全量 p4-BC（ep60 热启动 + God-AI 蒸馏）· Colab

在 **`curricula/p4-onset.jsonc`**（4 敌混编 basic/fast/power/armor、player 正中、
敌人四角，hard / 1 命 0 星 / 2400 ticks）上，从 **p1 ep60 权重**（`battle-p1bc-ep60.zip`）
warm-start，用 p4 语料（`p4-godai.zip`，God-AI 在 p4 课程上的胜局 + 濒危帧）训练
**`nn-training/train/bc.py --arch student --value-coef 0.5`** 共 **60 epoch**，
支持随时中断 / 断线续跑（进度与权重落 **Google Drive**），并可从任意 checkpoint
跑 **100 局贪心评估**（工具 `tools/sim/eval-course-ckpt.ts`）。

## 背景（为什么是 p4-BC，而不是直接 RL）

- p1 单敌已被 ep60 蒸馏饱和：94/100 ≈ 教师 92/100（docs/rl.progress.md §1）。
- p4 是泛化体检：ep60 直接评估 **14/100** vs 教师 **64/100** —— 学生的失败形态是
  「每局杀 1.9/4 但 74 局先被打死」（覆盖缺口：ep60 没见过多敌场景）。
- 判定规则（DECISIONS §327）：p4-BC 若 >= ~55-60% = 接近教师、p4 达标；若 <40%
  平台期 = 蒸馏到头，转入 RL（届时 p4-BC 权重仍是最佳 RL 起点）。

## 运行前

1. **Runtime → Change runtime type → T4 GPU**（强烈建议；CPU 单 epoch 40-90 分钟不现实）。
2. 备好两个文件：
   - `p4-godai.zip`（本机 `tmp/p4-godai.zip`，约 10-20 MB，顶层目录 `p4-godai-v2/`）
   - `battle-p1bc-ep60.zip`（p1 训练的 ep60 权重，你已下载到本机）
3. 建议挂载 Drive：`MyDrive/battle2-p4bc/` 存权重与进度，重启 runtime 后重跑
   第 1、5 节即可续训。
4. 仓库 clone 需要匿名 HTTPS 可读。私有仓库：`https://<token>@github.com/HuangJian/battle.git`
   （token 勿提交）。

## Cell 地图

| 节 | 作用 | 可重复 |
|---|---|---|
| 0 | 配置 | 是 |
| 1 | 挂 Drive + clone | 是 |
| 2 | python 工具链 + bun（首次 ~3-6 分钟装 torch） | 是（幂等） |
| 3 | 解压 p4 语料 + ep60 热启动权重（首次弹上传框） | 是 |
| 4 | 训练驱动函数（进度落 Drive） | 是 |
| **5** | **启动训练 + 跟随**（中断即停 cell；续跑重跑本 cell） | 是 |
| 6 | 彻底停止后台训练（可选） | 是 |
| 7 | 选择评估权重 | 是 |
| **8** | 100 局评估 | 是 |
| 9 | 汇总打印 | 是 |

> resume 语义（DECISIONS §325）：bc.py `--resume` 是**权重 warm-start** ——
> AdamW / 余弦 LR 不随权重延续。首启从 ep60 warm-start 后 LR 走完整余弦(60)；
> 中断续跑时 LR 从头再来（至多丢进程尾段 <1 epoch）。尽量一次跑长。


In [ ]:
# ============================================================
# 0) 配置（可编辑）
# ============================================================
import os, json, subprocess, sys, time, glob, shutil, re, zipfile
from pathlib import Path

REPO_URL   = "https://github.com/HuangJian/battle.git"   # 仓库（Colab 需可匿名访问；私有仓库见下方说明）
BRANCH     = "goal-nn"                                   # 训练代码所在分支
REPO_DIR   = Path("/content/battle2")
COURSE     = "p4-onset"                                  # 课程名 -> nn-training/curricula/p4-onset.jsonc
CORPUS_REL = "tmp/p4-godai-v2"                           # 语料解压目标（repo 相对）
ZIP_NAME   = "p4-godai.zip"                              # 你手动上传的 p4 语料压缩包文件名
WARM_ZIP   = "battle-p1bc-ep60.zip"                      # p1 ep60 权重压缩包（热启动，DECISIONS §325 resume）
TOTAL_EPOCHS = 60                                        # p4-BC 全量 epoch 数
SEED       = 1234                                        # 与 p1 全量一致的随机种子
DRIVE_DIR  = Path("/content/drive/MyDrive/battle2-p4bc") # Drive 上的持久目录（权重/进度）
MOUNT_DRIVE = True                                       # False = 只用 Colab 临时盘（重启丢进度）

# BC 超参（= p1 全量既定配置，p4 课程注释口径）
BC_ARCH, BC_VALUE_COEF = "student", 0.5
BC_BATCH, BC_LR, BC_VAL_SPLIT, BC_MIRROR, BC_NUM_WORKERS = 256, 0.003, 0.1, 0.5, 0

print("[0] 配置已加载；仓库目录 =", REPO_DIR)


## 预期输出（第 2 节）

```
[2] python/torch: 3.10.x torch 2.7.1 cuda True     # repo venv（bootstrap.py 探测 T4）
[2] DEVICE = cuda
```

> 若走 uv bootstrap（repo 规范路径），首次下载 ~2-3 GB cu 版 torch 约 3-6 分钟，
> 之后每个新 runtime 只需重跑一次。Colab 系统 python 已带 GPU torch 则直接复用。


In [ ]:
# ============================================================
# 1) 挂载 Drive + clone 仓库（幂等）
# ============================================================
def sh(args, **kw):
    """跑命令，返回 CompletedProcess；默认抛错。"""
    kw.setdefault("check", True)
    kw.setdefault("capture_output", True)
    kw.setdefault("text", True)
    return subprocess.run(args, **kw)

if MOUNT_DRIVE:
    try:
        from google.colab import drive  # type: ignore
        drive.mount("/content/drive")
    except Exception as e:
        print("[1] Drive 挂载失败（仍继续，进度只在本机）:", e)
    DRIVE_DIR.mkdir(parents=True, exist_ok=True)
    print("[1] Drive 持久目录:", DRIVE_DIR)

if (REPO_DIR / ".git").exists() and not shutil.which("git") is None:
    print("[1] 仓库已存在 -> git pull")
    sh(["git", "-C", str(REPO_DIR), "fetch", "--depth", "1", "origin", BRANCH])
    sh(["git", "-C", str(REPO_DIR), "checkout", "-B", BRANCH, "FETCH_HEAD"])
else:
    print("[1] 全新 clone（shallow, branch=%s）..." % BRANCH)
    REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
    sh(["git", "clone", "--depth", "1", "--single-branch", "--branch", BRANCH, REPO_URL, str(REPO_DIR)])

sha = sh(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"]).stdout.strip()
print("[1] repo HEAD:", sha[:12], "at", REPO_DIR)


In [ ]:
# ============================================================
# 2) python 工具链 + bun（评估需要）
#    优先 repo 自带 uv venv（bootstrap.py，规范路径）；没有则退回
#    Colab 系统 python（自带 GPU torch）；再不行才跑 uv bootstrap 装。
# ============================================================
NN = REPO_DIR / "nn-training"
VENV_PY = NN / ".venv" / "bin" / "python"

def py_ok(py: str) -> bool:
    try:
        r = sh([py, "-c", "import torch, numpy"], check=False, capture_output=True)
        return r.returncode == 0
    except Exception:
        return False

PY = None
if VENV_PY.exists() and py_ok(str(VENV_PY)):
    PY = str(VENV_PY)
    print("[2] 使用 repo venv python:", PY)
else:
    sys_py = shutil.which("python3") or shutil.which("python")
    if sys_py and py_ok(sys_py):
        PY = sys_py
        print("[2] venv 缺失/不可用 -> 使用系统 python（自带 torch）:", PY)
    else:
        print("[2] 无 torch -> 跑 uv bootstrap.py（探测 GPU -> uv sync -> 自检）")
        sh([sys_py or "python3", str(NN / "bootstrap.py")], cwd=str(NN))
        assert VENV_PY.exists(), "bootstrap 未产出 venv"
        PY = str(VENV_PY)
        print("[2] bootstrap 完成:", PY)

ver = sh([PY, "-c", "import sys, torch; print(sys.version.split()[0], 'torch', torch.__version__, 'cuda', torch.cuda.is_available())"]).stdout.strip()
print("[2] python/torch:", ver)
DEVICE = "cuda" if sh([PY, "-c", "import torch; print(torch.cuda.is_available())"]).stdout.strip() == "True" else "cpu"
print("[2] DEVICE =", DEVICE)

# --- bun（评估工具 tools/sim/eval-course-ckpt.ts 需要）---
BUN = None
for cand in (os.path.expanduser("~/.bun/bin/bun"), shutil.which("bun")):
    if cand and os.path.exists(cand):
        BUN = cand
        break
if BUN is None:
    print("[2] 安装 bun ...")
    sh(["bash", "-lc", "curl -fsSL https://bun.sh/install | bash"])
    BUN = os.path.expanduser("~/.bun/bin/bun")
print("[2] bun:", BUN, sh([BUN, "--version"]).stdout.strip())

# 依赖守卫：bc.py --device 与 eval-course-ckpt（DECISIONS §325 / §326）
need = [NN / "train" / "bc.py", REPO_DIR / "tools" / "sim" / "eval-course-ckpt.ts"]
missing = [str(p) for p in need if not p.exists()]
assert not missing, (
    "clone 的仓库缺少: " + ", ".join(missing) +
    " —— 需先把 bc.py --device 与 tools/sim/eval-course-ckpt.ts 的提交推送到 " + BRANCH + " 分支"
)
print("[2] 依赖文件齐全（bc.py --device + eval-course-ckpt）")


In [ ]:
# ============================================================
# 3) 解压语料（幂等）+ 准备 ep60 热启动权重
#    语料 zip 优先 Drive 上已存的，否则弹上传框；WARM_ZIP 同理
# ============================================================
def count_shards(d: Path) -> int:
    if not d.is_dir():
        return 0
    n = 0
    for sub in d.iterdir():
        if sub.is_dir() and (sub / "obs.npy").exists():
            n += 1
    return n

def extract_zip(src_zip: Path, dest_root: Path, min_shards: int, label: str) -> Path:
    """解压 zip 到 dest_root 下临时目录，返回含 >= min_shards 个 shard 的目录。"""
    tmp_root = dest_root / (".extract-" + label)
    if tmp_root.exists():
        shutil.rmtree(tmp_root)
    tmp_root.mkdir(parents=True)
    print(f"[3] 解压 {label}（{src_zip.stat().st_size // (1024*1024)} MB）...")
    with zipfile.ZipFile(src_zip) as z:
        z.extractall(tmp_root)
    tops = [p for p in tmp_root.iterdir()]
    payload = None
    if len(tops) == 1 and tops[0].is_dir() and count_shards(tops[0]) >= min_shards:
        payload = tops[0]
    elif count_shards(tmp_root) >= min_shards:
        payload = tmp_root
    assert payload is not None, f"zip 里找不到含 obs.npy 的 shard 目录（{label}）"
    shutil.rmtree(tmp_root, ignore_errors=True)
    return payload

# ---- 语料 ----
CORPUS = REPO_DIR / CORPUS_REL
n = count_shards(CORPUS)
print("[3] 现有语料 shard 数:", n, "at", CORPUS)
if n < 1000:
    zip_candidates = []
    if MOUNT_DRIVE:
        zip_candidates.append(DRIVE_DIR / ZIP_NAME)
    zip_candidates.append(REPO_DIR / "tmp" / ZIP_NAME)
    src_zip = next((z for z in zip_candidates if z.exists()), None)
    if src_zip is None:
        print("[3] 请在弹出的上传框选择本机文件:", ZIP_NAME, "(约 10-20 MB，p4 混编语料)")
        from google.colab import files  # type: ignore
        up = files.upload()
        fname = next(iter(up))
        dest = (DRIVE_DIR if MOUNT_DRIVE else REPO_DIR / "tmp") / ZIP_NAME
        dest.parent.mkdir(parents=True, exist_ok=True)
        dest.write_bytes(up[fname])
        src_zip = dest
        print("[3] 已保存上传:", src_zip, src_zip.stat().st_size, "bytes")
    payload = extract_zip(src_zip, REPO_DIR / "tmp", 1000, "corpus")
    CORPUS.parent.mkdir(parents=True, exist_ok=True)
    if CORPUS.exists():
        shutil.rmtree(CORPUS)
    os.rename(str(payload), str(CORPUS))
    n = count_shards(CORPUS)
    print("[3] 语料就位:", n, "shards")

sample = next((s for s in CORPUS.iterdir() if (s / "obs.npy").exists()), None)
if sample:
    files = sorted(p.name for p in sample.iterdir())
    print("[3] 样例 shard 文件:", files)
    assert "obs.npy" in files and "actions.npy" in files, "shard 缺关键 npy"
print("[3] corpus OK ->", CORPUS, "shards =", n)

# ---- ep60 热启动权重（p1 训练产物，warm-start 到 p4）----
WARM_DIR = Path("/content/warmstart")
WARM_DIR.mkdir(parents=True, exist_ok=True)
def find_weights_json(d: Path) -> Path | None:
    # 优先 weights.json（active pointer），否则最大编号 ckpt，否则任何 *.json 权重
    active = d / "weights.json"
    if active.exists():
        return active
    ckpts = sorted(d.glob("weights.json.ckpt.*"),
                   key=lambda p: int(re.search(r"\.ckpt\.(\d+)$", p.name).group(1)))
    if ckpts:
        return ckpts[-1]
    for p in sorted(d.rglob("*.json")):
        if p.name.startswith("weights"):
            return p
    return None

WARM_WEIGHTS = None
# 1) 已解压的（上次运行残留）
w = find_weights_json(WARM_DIR)
if w is not None:
    WARM_WEIGHTS = w
    print("[3] 热启动权重（已就位）:", WARM_WEIGHTS)
# 2) Drive / repo 上直接有 weights json
if WARM_WEIGHTS is None:
    for cand in (DRIVE_DIR / "ep60.json", REPO_DIR / "tmp" / "ep60.json"):
        if cand.exists():
            WARM_WEIGHTS = cand
            print("[3] 热启动权重（直接 json）:", WARM_WEIGHTS)
            break
# 3) WARM_ZIP（battle-p1bc-ep60.zip）-> 解压
if WARM_WEIGHTS is None:
    warm_zip = None
    for cand in ([DRIVE_DIR / WARM_ZIP] if MOUNT_DRIVE else []) + [REPO_DIR / "tmp" / WARM_ZIP]:
        if cand.exists():
            warm_zip = cand
            break
    if warm_zip is None:
        print("[3] 请在弹出的上传框选择 p1 ep60 权重压缩包:", WARM_ZIP)
        from google.colab import files  # type: ignore
        up = files.upload()
        fname = next(iter(up))
        dest = (DRIVE_DIR if MOUNT_DRIVE else REPO_DIR / "tmp") / WARM_ZIP
        dest.parent.mkdir(parents=True, exist_ok=True)
        dest.write_bytes(up[fname])
        warm_zip = dest
        print("[3] 已保存上传:", warm_zip, warm_zip.stat().st_size, "bytes")
    tmp_w = WARM_DIR / ".warm-extract"
    if tmp_w.exists():
        shutil.rmtree(tmp_w)
    tmp_w.mkdir(parents=True)
    with zipfile.ZipFile(warm_zip) as z:
        z.extractall(tmp_w)
    w = find_weights_json(tmp_w)
    assert w is not None, "WARM_ZIP 里找不到 weights json"
    shutil.copy(w, WARM_DIR / "ep60.json")
    shutil.rmtree(tmp_w, ignore_errors=True)
    WARM_WEIGHTS = WARM_DIR / "ep60.json"
    print("[3] 热启动权重（解压自 zip）:", WARM_WEIGHTS)

if WARM_WEIGHTS is not None:
    meta = json.loads(WARM_WEIGHTS.read_text())
    print("[3] warm-start 权重 arch =", meta.get("arch"), "| schema =", meta.get("schema_major"))
assert WARM_WEIGHTS is not None, "必须提供 ep60 热启动权重（WARM_ZIP 或直接 json）"


In [ ]:
# ============================================================
# 4) 训练驱动：连续全量 run + 每 epoch 落盘 ckpt -> Drive，随时可停/续
#
# 持久状态 = DRIVE_DIR/run_state.json：{total, done, base, running, pid, repo_sha}
# 训练进程以 start_new_session 后台运行（monitor cell 停止不杀训练）。
# resume 语义（DECISIONS §325）：bc.py --resume 是权重 warm-start —— 首次启动
# 从 ep60 权重 warm-start（base==0 且 WARM_WEIGHTS 就位），中断后续跑从
# run/resume_seed.json（p4 自己的进度）。
# ============================================================
RUN = DRIVE_DIR / "run"          # bc.py 的 --out 目录
LOG = Path("/content/p4bc-train.log")
STATE = DRIVE_DIR / "run_state.json"
RUN.mkdir(parents=True, exist_ok=True)
RUN_PROC = None                   # 当前后台训练进程（Popen）

def load_state() -> dict:
    if STATE.exists():
        try:
            return json.loads(STATE.read_text())
        except Exception:
            pass
    return {"total": TOTAL_EPOCHS, "done": 0, "base": 0, "running": False, "pid": None, "repo_sha": None}

def save_state(s: dict) -> None:
    STATE.write_text(json.dumps(s, indent=1))

def latest_ckpt_n() -> int:
    best = 0
    for f in RUN.glob("weights.json.ckpt.*"):
        m = re.search(r"\.ckpt\.(\d+)$", f.name)
        if m:
            best = max(best, int(m.group(1)))
    return best

def bank_progress(s: dict) -> dict:
    """把当前进程已写出的最高 ckpt 计入 done，并刷新 resume_seed.json。"""
    k = latest_ckpt_n()
    if k > 0:
        eff = s["base"] + k
        if eff > s["done"]:
            s["done"] = eff
            src = RUN / f"weights.json.ckpt.{k}"
            if src.exists():
                tmp = RUN / ".resume_seed.tmp"
                shutil.copy(src, tmp)
                os.replace(tmp, RUN / "resume_seed.json")
            save_state(s)
    return s

def running_proc():
    """返回在跑的训练 Popen（无则 None）。"""
    global RUN_PROC
    if RUN_PROC is not None and RUN_PROC.poll() is not None:
        RUN_PROC = None              # 已退出但没被收尾 -> 清掉
    return RUN_PROC

def settle_train(rc: int) -> None:
    """进程退出后的收尾：rc==0 => 全部完成；否则按已落盘 ckpt 记账。"""
    s = load_state()
    if rc == 0:
        s["done"] = s["total"]
    else:
        s = bank_progress(s)
    s["running"] = False
    s["pid"] = None
    save_state(s)
    print(f"[monitor] 训练进程退出 rc={rc} | done={s['done']}/{s['total']} "
          + ("ALL DONE" if s["done"] >= s["total"] else "-> 重跑第 5 节续训"))
    try:
        for l in LOG.read_text(errors="replace").splitlines()[-5:]:
            print("[log]", l[:200])
    except Exception:
        pass

def launch_train() -> int:
    global RUN_PROC
    if running_proc() is not None:
        print(f"[train] 已有训练在跑 pid={RUN_PROC.pid}（重跑第 5 节可挂上 monitor）")
        return RUN_PROC.pid
    s = load_state()
    base = s.get("done", 0)
    remaining = s["total"] - base
    if remaining <= 0:
        print(f"[train] 已完成 {base}/{s['total']} epoch，无需再跑")
        return 0
    # 清理上一个（已死）进程的临时 ckpt，保留 weights.json / resume_seed.json
    for f in RUN.glob("weights.json.ckpt.*"):
        f.unlink()
    args = [PY, "-u", "train/bc.py",
            "--data-dir", str(CORPUS),
            "--out", str(RUN / "weights.json"),
            "--arch", BC_ARCH, "--value-coef", str(BC_VALUE_COEF),
            "--epochs", str(remaining), "--ckpt-every", "1",
            "--batch", str(BC_BATCH), "--lr", str(BC_LR),
            "--val-split", str(BC_VAL_SPLIT), "--mirror-p", str(BC_MIRROR),
            "--num-workers", str(BC_NUM_WORKERS),
            "--seed", str(SEED), "--device", DEVICE,
            "--notes", "colab p4-bc %s/%s seed=%s" % (base + 1, s["total"], SEED)]
    resume = RUN / "resume_seed.json"
    if base == 0 and WARM_WEIGHTS is not None:
        args += ["--resume", str(WARM_WEIGHTS)]          # ep60 热启动（首启）
        print("[train] warm-start from ep60:", WARM_WEIGHTS)
    elif base > 0 and resume.exists():
        args += ["--resume", str(resume)]                # p4 进度续跑
    print("[train] launch:", " ".join(str(a) for a in args))
    LOG.parent.mkdir(parents=True, exist_ok=True)
    logf = open(LOG, "ab")
    RUN_PROC = subprocess.Popen(args, cwd=str(NN), stdout=logf, stderr=subprocess.STDOUT,
                                start_new_session=True)
    s.update({"running": True, "pid": RUN_PROC.pid, "base": base,
              "repo_sha": sh(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"],
                             check=False).stdout.strip()})
    save_state(s)
    print(f"[train] pid={RUN_PROC.pid} base={base} -> target {s['total']} (log: {LOG})")
    return RUN_PROC.pid

def monitor(interval: float = 20.0):
    p = running_proc()
    if p is None:
        print("[monitor] 没有在跑的训练（先跑第 5 节 launch）。done =",
              load_state().get("done"))
        return
    t0 = time.time()
    pos = 0
    last_print = 0
    while p.poll() is None:
        s = bank_progress(load_state())
        try:
            txt = LOG.read_text(errors="replace")
            for line in txt[pos:].splitlines():
                if "epoch" in line.lower() and "loss" in line:
                    print("[train]", line.strip()[:160])
            pos = len(txt)
        except Exception:
            pass
        el = int(time.time() - t0)
        if el - last_print >= 120:
            last_print = el
            print(f"[monitor] {el}s elapsed | banked {s['done']}/{s['total']} epochs")
        time.sleep(interval)
    settle_train(p.wait())


In [ ]:
# ============================================================
# 5) 启动训练并跟随（阻塞式 monitor）
#    * 首次运行：从 ep60 权重 warm-start，训 60 epoch（p4 语料）
#    * 中断/断线后续跑：先跑本 cell（自动从 Drive 最近进度 resume）
#    * 想中断本 cell 但保留训练：点 ■ —— 训练进程在后台继续，重跑本 cell 挂上 monitor
# ============================================================
launch_train()
monitor()


In [ ]:
# ============================================================
# 6) （可选）彻底停止后台训练进程
# ============================================================
def stop_train():
    global RUN_PROC
    p = RUN_PROC
    if p is None:
        print("[stop] 无运行中的训练进程")
        return
    if p.poll() is not None:
        settle_train(p.returncode)
        RUN_PROC = None
        print("[stop] 进程此前已退出（rc=%s），状态已收尾" % p.returncode)
        return
    pid = p.pid
    p.terminate()
    try:
        p.wait(timeout=15)
    except subprocess.TimeoutExpired:
        p.kill()
        p.wait()
    settle_train(p.returncode)
    RUN_PROC = None
    print("[stop] 已停止 pid", pid)

stop_train()
print("[stop] done =", load_state().get("done"), "/", TOTAL_EPOCHS)


## 中断 / 续跑怎么用

- **停 cell 但让训练继续**：点 monitor cell 的 ■。`start_new_session` 保证训练
  不受影响 —— 重跑第 5 节只是重新挂上 monitor。
- **彻底停训练**：跑第 6 节（SIGTERM → SIGKILL）。已完成 epoch 已按每 epoch 落盘
  并计入 `run_state.json`。
- **Colab runtime 被回收 / 断线**：重启后依次重跑 0→1→2→3（幂等）→5。第 5 节
  自动从 Drive 的 `resume_seed.json` 续跑剩余 epoch（首启已用过的 ep60 warm-start
  不会重复——base>0 时只从 p4 自己的进度续）。
- **全部完成**：monitor 打印 `ALL DONE`；`run/weights.json` = 末段 best。


In [ ]:
# ============================================================
# 7) 选择要评估的 checkpoint（默认 = 最近完成进程的 best：run/weights.json）
# ============================================================
s = load_state()
print(f"[pick] 训练进度: done={s['done']}/{s['total']}")

def list_weights():
    out = []
    for f in sorted(RUN.glob("weights.json")):
        out.append(f)
    for f in sorted(RUN.glob("weights.*.json"), reverse=True):
        out.append(f)
    return out

cands = list_weights()
seen, uniq = set(), []
for c in cands:
    if c.name not in seen:
        seen.add(c.name)
        uniq.append(c)
print("[pick] 候选权重文件:")
for i, c in enumerate(uniq):
    size = c.stat().st_size // 1024
    print(f"  [{i}] {c.name}  ({size} KB)")

EVAL_WEIGHTS = [uniq[0]] if uniq else []
print("[pick] EVAL_WEIGHTS =", [w.name for w in EVAL_WEIGHTS])


In [ ]:
# ============================================================
# 8) 从 checkpoint 跑 100 局评估（tools/sim/eval-course-ckpt.ts，贪心部署口径）
#    p4-onset 单关 × seeds 0..99；对比基线：God-AI 64/100、ep60 14/100
#    （docs/rl.progress.md §2，本机验证口径）
# ============================================================
N_GAMES = 100          # p4 单关 × 100 seeds（seed = 0..99）
EVAL_WORKERS = max(1, (os.cpu_count() or 2) - 1)

assert EVAL_WEIGHTS, "先跑第 7 节选择权重（或手动设 EVAL_WEIGHTS = [Path(...)]）"
print(f"[eval] {N_GAMES} games x {len(EVAL_WEIGHTS)} weights, workers={EVAL_WORKERS}")

rows_out = Path("/content/p4-eval-rows.jsonl")
cmd = [BUN, "tools/sim/eval-course-ckpt.ts",
       "--course", str(NN / "curricula" / (COURSE + ".jsonc")),
       "--games", str(N_GAMES), "--workers", str(EVAL_WORKERS),
       "--out", str(rows_out)]
for w in EVAL_WEIGHTS:
    cmd += ["--weights", str(w)]
print("[eval]", " ".join(cmd))
t0 = time.time()
r = subprocess.run(cmd, cwd=str(REPO_DIR), capture_output=True, text=True)
print(r.stderr)
print(f"[eval] exit={r.returncode} in {time.time()-t0:.0f}s rows={len(rows_out.read_text().splitlines()) if rows_out.exists() else 0}")
assert r.returncode == 0, "评估失败"


In [ ]:
# ============================================================
# 9) 评估汇总（每权重 胜率/击杀/击中/被击中）
# ============================================================
rows = [json.loads(l) for l in rows_out.read_text().splitlines()]
print(f"[summary] {len(rows)} 局")

import collections
by = collections.defaultdict(list)
for r in rows:
    by[r["label"]].append(r)
hdr = f'{"权重":<34}{"胜":>6}{"杀":>5}{"击中":>6}{"被击中":>7}{"损血":>8}{"开枪":>7}{"超时":>7}{"死亡":>6}'
print(hdr)
for lab, g in sorted(by.items()):
    wins = sum(1 for r in g if r["win"])
    outs = collections.Counter(r["outcome"] for r in g)
    print(f'{lab:<34}{str(wins)+"/"+str(len(g)):>7}{sum(r["kills"] for r in g):>6}'
          f'{sum(r["enemyHits"] for r in g):>7}{sum(r["playerHits"] for r in g):>8}'
          f'{sum(r["playerDamageTaken"] for r in g):>9}{sum(r["playerShots"] for r in g):>8}'
          f'{outs.get("max_ticks", 0):>8}{outs.get("gameover", 0):>7}')
print()
print("对比基线（docs/rl.progress.md §2，同一工具/口径）：")
print("  God-AI 64/100（kills 306，被击中 34）· ep60（p1 权重直接评估）14/100（kills 193，被击中 75）")
print("  判读规则：p4-BC 若 >= ~55-60% = 接近教师（p4 达标）；<40% 平台期 = 需 RL")


## 评估怎么用

第 7 节列出候选权重（active `weights.json` = 最近完成段的 best），改 `EVAL_WEIGHTS`
可多选对比，然后跑第 8、9 节。评估口径 = RL eval 同一路径
（`export-eval-game.runEvalOne`，掩码 argmax 贪心、课程 lives/level 覆盖），
100 局 = p4 单关 × seeds 0..99（确定性）。

字段：`win` 胜率、`kills`、`enemyHits` 击中、`playerHits`(死亡+星盾) +
`playerDamageTaken`(非致命扣血) 被击中。对比基线：God-AI 64/100、ep60 14/100
（docs/rl.progress.md §2，同一工具/口径，可复跑：`--policy god`）。
